# Forecasting com Prophet

Escolhi Prophet como primeiro modelo porque ele lida nativamente com sazonalidade múltipla (semanal, anual) e é robusto a dados faltantes e outliers, dois problemas que ficaram evidentes na EDA. Além disso, o ajuste de feriados e eventos especiais é simples de implementar, o que importa bastante para varejo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Preparando os dados para o Prophet

O Prophet exige um formato específico: coluna `ds` para a data e `y` para o valor previsto. Vou trabalhar primeiro com a demanda agregada de toda a base, depois mostro como escalar para SKUs individuais.

In [ ]:
df = pd.read_parquet('../data/processed/features_modelagem.parquet')

# Agrego para a série temporal total, útil para entender o padrão geral antes de descer para SKU
df_total = df.groupby('data')['quantidade'].sum().reset_index()
df_total.columns = ['ds', 'y']
df_total = df_total.sort_values('ds')

print(f'Série temporal: {len(df_total)} dias')
print(f'Período: {df_total["ds"].min()} a {df_total["ds"].max()}')

## 2. Split treino/teste

Para séries temporais, não posso fazer split aleatório, sempre uso os últimos N períodos como teste para simular previsão real.

In [ ]:
# Uso os últimos 60 dias como teste
corte = df_total['ds'].max() - pd.Timedelta(days=60)
treino = df_total[df_total['ds'] <= corte]
teste = df_total[df_total['ds'] > corte]

print(f'Treino: {len(treino)} dias')
print(f'Teste: {len(teste)} dias')

## 3. Treinando o Prophet

Incluo sazonalidade semanal e anual. Desativo a sazonalidade diária porque já tenho dados diários agregados, não faz sentido aqui.

In [ ]:
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',  # Multiplicativo porque a sazonalidade cresce com a tendência
    changepoint_prior_scale=0.05,       # Controle de overfitting em mudanças de tendência
    interval_width=0.95
)

# Adiciono feriados do Reino Unido, dataset é de varejista britânico
modelo.add_country_holidays(country_name='UK')

modelo.fit(treino)
print('Modelo treinado com sucesso')

## 4. Gerando previsões e avaliando

In [ ]:
# Crio o dataframe futuro incluindo o período de teste
futuro = modelo.make_future_dataframe(periods=90)
previsao = modelo.predict(futuro)

# Métricas no período de teste
prev_teste = previsao[previsao['ds'].isin(teste['ds'])][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
resultado = teste.merge(prev_teste, on='ds')

mae = np.mean(np.abs(resultado['y'] - resultado['yhat']))
mape = np.mean(np.abs((resultado['y'] - resultado['yhat']) / resultado['y'])) * 100
rmse = np.sqrt(np.mean((resultado['y'] - resultado['yhat'])**2))

print(f'MAE:  {mae:.0f} unidades')
print(f'MAPE: {mape:.1f}%')
print(f'RMSE: {rmse:.0f} unidades')

In [ ]:
# Visualização da previsão vs real
fig, ax = plt.subplots()

ax.plot(df_total['ds'], df_total['y'], label='Demanda real', color='steelblue', alpha=0.7)
ax.plot(previsao['ds'], previsao['yhat'], label='Previsão Prophet', color='tomato', linewidth=2)
ax.fill_between(
    previsao['ds'], previsao['yhat_lower'], previsao['yhat_upper'],
    alpha=0.15, color='tomato', label='Intervalo de confiança 95%'
)
ax.axvline(corte, color='gray', linestyle='--', label='Início do teste')

ax.set_title('Previsão de demanda, Prophet')
ax.set_xlabel('Data')
ax.set_ylabel('Quantidade vendida')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/prophet_previsao.png', dpi=150)
plt.show()

## 5. Componentes da previsão

Uma das vantagens do Prophet é a decomposição explícita em tendência, sazonalidade anual e semanal. Isso ajuda a comunicar resultados para times de negócio sem precisar abrir o modelo.

In [ ]:
fig_comp = modelo.plot_components(previsao)
plt.tight_layout()
plt.savefig('../reports/figures/prophet_componentes.png', dpi=150)
plt.show()

## 6. Validação cruzada temporal

Uso cross-validation temporal do Prophet para ter uma estimativa mais robusta de erro. Isso simula N retreinamentos em janelas deslizantes de tempo.

In [ ]:
# initial: período mínimo de treino | horizon: janela de previsão | period: intervalo entre avaliações
cv_result = cross_validation(
    modelo,
    initial='365 days',
    period='30 days',
    horizon='60 days',
    parallel='processes'
)

metricas_cv = performance_metrics(cv_result)
print('Métricas de cross-validation temporal:')
print(metricas_cv[['horizon', 'mae', 'mape', 'rmse']].tail(10).to_string(index=False))